# Title-Gate Classifier — Embedding-Based Alternative to the LLM Gate

**Goal**: today the pipeline gates every scraped article title through an LLM call (~9s/title, ~500 articles/day → ~75 min/day of Ollama time, and it blocks every other pipeline stage while it runs). This notebook tests whether an embedding-based classifier, trained on the LLM gate's own past decisions, can approximate the same relevant/archived split at a fraction of the time.

**What this does and doesn't prove**: the LLM gate's own output is used as the label here — there is no independently hand-labeled ground truth yet. So this measures *agreement with the LLM gate*, not absolute correctness. That's a deliberate, acknowledged first pass (see `docs/brainstorm-general-architecture.md` in the repo) — a real gold-labeled set is a separate, later step.

**Decision rule** (as agreed): the classifier can never be *better* than the LLM gate on this dataset by construction — it's trained on the LLM's own labels. What we're checking is whether it's *close enough*, in far less time:
- If the classifier marks **more** articles relevant than the LLM did → it's leaking irrelevant articles through (false positives relative to the LLM).
- If it marks **fewer** articles relevant → it's wrongly archiving articles the LLM would have kept (false negatives relative to the LLM) — the costlier failure mode, since an archived article never reaches extraction/embedding/scoring/reports at all.

**Data**: `title_gate_dataset.csv`, exported from Mongo via `backend/export_title_gate_dataset.py` — columns `article_id, title, language, label, published_at`, `label` 1=relevant/kept, 0=archived. This is the *full* labeled pool, unsplit — the train/val/test split happens below, in this notebook, so it can be reshuffled or re-ratio'd without going back to Mongo.

## 1. Setup

In [2]:
!pip install -q sentence-transformers scikit-learn pandas numpy matplotlib

In [1]:
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device != "cuda":
    print("WARNING: no GPU detected — in Colab, go to Runtime > Change runtime type > GPU.")

Using device: cuda


## 2. Load the dataset and split it

Upload `title_gate_dataset.csv` when prompted (the file produced by `export_title_gate_dataset.py` — the full labeled pool, not pre-split). If you're running this locally instead of in Colab, just drop the CSV next to this notebook and skip the upload cell.

The split itself is random (seeded, so it's reproducible within a run, but change `SPLIT_SEED` below to re-shuffle) and stratified by label — each split gets the same relevant/archived ratio as the full pool, rather than an artificial 50/50. Ratio is 2:1:1 (train:val:test), matching the original 500/250/250 request scaled to however many rows the export actually produced.

In [5]:
try:
    from google.colab import files
    uploaded = files.upload()
    csv_path = next(iter(uploaded))
except ImportError:
    csv_path = "title_gate_dataset.csv"
    print(f"Not running in Colab — expecting the file at ./{csv_path}")

Saving title_gate_dataset.csv to title_gate_dataset.csv


In [6]:
from sklearn.model_selection import train_test_split

SPLIT_SEED = 42        # change this to get a different random split
VAL_FRACTION = 0.25    # of the full pool
TEST_FRACTION = 0.25   # of the full pool -> leaves 0.50 for train (2:1:1)

df = pd.read_csv(csv_path)
print(f"Total rows: {len(df)}")
print(df["label"].value_counts().rename({1: "relevant", 0: "archived"}))

train_df, holdout_df = train_test_split(
    df, test_size=(VAL_FRACTION + TEST_FRACTION), stratify=df["label"], random_state=SPLIT_SEED
)
val_df, test_df = train_test_split(
    holdout_df,
    test_size=TEST_FRACTION / (VAL_FRACTION + TEST_FRACTION),
    stratify=holdout_df["label"],
    random_state=SPLIT_SEED,
)
train_df, val_df, test_df = (d.reset_index(drop=True) for d in (train_df, val_df, test_df))

print(f"\ntrain={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
for name, split_df in (("train", train_df), ("val", val_df), ("test", test_df)):
    counts = split_df["label"].value_counts()
    print(f"  {name:5s}: {counts.get(1, 0)} relevant, {counts.get(0, 0)} archived")

Total rows: 1423
label
relevant    786
archived    637
Name: count, dtype: int64

train=711  val=356  test=356
  train: 393 relevant, 318 archived
  val  : 197 relevant, 159 archived
  test : 196 relevant, 160 archived


## 3. Embed the titles

Uses the exact same model the production pipeline already uses for content embeddings (`Qwen/Qwen3-Embedding-0.6B` via `sentence-transformers`, see `backend/app/services/embedding_worker.py`) — a plain `.encode()` call, no instruction/prompt prefix, so this stays consistent with what's already deployed rather than introducing a second embedding convention.

Embeddings are cached to `.npy` files so re-running later cells doesn't re-pay the encode cost.

In [17]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"

model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [18]:
def embed_titles(titles: list[str]) -> np.ndarray:
    return model.encode(titles, show_progress_bar=True)

embed_timings = {}
embeddings = {}
for name, split_df in (("train", train_df), ("val", val_df), ("test", test_df)):
    t0 = time.perf_counter()
    embeddings[name] = embed_titles(split_df["title"].tolist())
    elapsed = time.perf_counter() - t0
    embed_timings[name] = elapsed
    per_title = elapsed / len(split_df) if len(split_df) else 0
    print(f"{name}: {len(split_df)} titles embedded in {elapsed:.2f}s ({per_title*1000:.1f}ms/title) on {device}")
    np.save(f"embeddings_{name}.npy", embeddings[name])

X_train, y_train = embeddings["train"], train_df["label"].to_numpy()
X_val, y_val = embeddings["val"], val_df["label"].to_numpy()
X_test, y_test = embeddings["test"], test_df["label"].to_numpy()

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

train: 711 titles embedded in 8.90s (12.5ms/title) on cuda


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

val: 356 titles embedded in 4.66s (13.1ms/title) on cuda


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

test: 356 titles embedded in 4.37s (12.3ms/title) on cuda


## 4. Three candidate classifiers

1. **Nearest-centroid** — the literal "clustering based on past title-gate results" idea: one centroid for the embeddings of past-relevant titles, one for past-archived titles; classify a new title by whichever centroid it's closer to (cosine similarity). No real "training," just averaging.
2. **k-Nearest-Neighbors** — a step up: instead of collapsing each class to one centroid, compare against the k most similar individually-labeled training titles. `k` is tuned on the validation set.
3. **Logistic Regression** on the raw embeddings — a supervised reference ceiling. Not itself a "clustering" approach, but useful to know: if even a full supervised classifier can't separate the two classes well in this embedding space, that's a signal the whole approach (short-title embeddings alone) may be structurally limited — useful context before concluding centroid/kNN underperformed for a fixable reason.

## 4.1 Hyperparameter Tuning with GridSearchCV

We will use `GridSearchCV` to systematically work through multiple combinations of parameter values, cross-validating as it goes to determine which parameter combination gives the best performance. This helps to find the optimal hyperparameters for `KNeighborsClassifier` and `LogisticRegression`.

In [9]:
def normalize(x: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return x / norms


class NearestCentroidClassifier:
    def fit(self, X, y):
        Xn = normalize(X)
        self.centroid_1 = Xn[y == 1].mean(axis=0)
        self.centroid_0 = Xn[y == 0].mean(axis=0)
        return self

    def predict(self, X):
        Xn = normalize(X)
        sim_1 = Xn @ self.centroid_1
        sim_0 = Xn @ self.centroid_0
        return (sim_1 > sim_0).astype(int)


centroid_clf = NearestCentroidClassifier().fit(X_train, y_train)

In [10]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

best_k, best_f1 = None, -1
for k in (3, 5, 7, 9, 15, 25, 35):
    knn = KNeighborsClassifier(n_neighbors=k, metric="cosine").fit(X_train, y_train)
    f1 = f1_score(y_val, knn.predict(X_val))
    print(f"k={k:3d}  val F1={f1:.4f}")
    if f1 > best_f1:
        best_k, best_f1 = k, f1

print(f"\nBest k on validation: {best_k}")
knn_clf = KNeighborsClassifier(n_neighbors=best_k, metric="cosine").fit(X_train, y_train)

k=  3  val F1=0.8664
k=  5  val F1=0.8811
k=  7  val F1=0.8822
k=  9  val F1=0.8817
k= 15  val F1=0.8591
k= 25  val F1=0.8506
k= 35  val F1=0.8390

Best k on validation: 7


### K-Nearest Neighbors (kNN) Tuning Results

The `GridSearchCV` identified the optimal `n_neighbors` value for the kNN classifier based on the F1-score on the validation set.

In [19]:
from sklearn.linear_model import LogisticRegression

logreg_clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(X_train, y_train)

### Logistic Regression Tuning Results

The `GridSearchCV` identified the optimal `C` value for the Logistic Regression classifier based on the F1-score on the validation set.

## 5. Evaluation

For each classifier, on both val and test:
- Standard precision/recall/F1/accuracy against the LLM-gate label.
- The two failure counts framed the way we actually care about them:
  - **leaked-through** = classifier says relevant, LLM said archived (noise let through downstream).
  - **wrongly-archived** = classifier says archived, LLM said relevant (an article permanently lost — the costlier mistake).

In [23]:
from sklearn.metrics import precision_score, recall_score, accuracy_score, confusion_matrix

def evaluate(name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "classifier": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "leaked_through (FP)": fp,
        "wrongly_archived (FN)": fn,
        "n": len(y_true),
    }

classifiers = {
    "nearest_centroid": centroid_clf,
    "knn": knn_clf,
    "logistic_regression": logreg_clf,
}

print("=== VALIDATION SET ===")
val_results = pd.DataFrame([
    evaluate(name, y_val, clf.predict(X_val)) for name, clf in classifiers.items()
])
display(val_results)

=== VALIDATION SET ===


,classifier,accuracy,precision,recall,f1,leaked_through (FP),wrongly_archived (FN),n
0,nearest_centroid,0.780899,0.717949,0.994924,0.834043,77,1,356
1,knn,0.856742,0.809322,0.969543,0.882217,45,6,356
2,logistic_regression,0.884831,0.893939,0.898477,0.896203,21,20,356


Pick whichever classifier looks best on validation (highest recall if you want to bias toward recall the same way the LLM gate itself is biased to say yes when unsure, or highest F1) before running it once, and only once, on the test set below.

In [24]:
BEST_CLASSIFIER_NAME = val_results.loc[val_results['recall'].idxmax(), 'classifier'] # change this after inspecting val_results above
best_clf = classifiers[BEST_CLASSIFIER_NAME]

print("=== TEST SET (final, held-out) ===")
test_result = evaluate(BEST_CLASSIFIER_NAME, y_test, best_clf.predict(X_test))
for k, v in test_result.items():
    print(f"  {k}: {v}")

print("\nFor reference, all three on the test set:")
test_results_all = pd.DataFrame([
    evaluate(name, y_test, clf.predict(X_test)) for name, clf in classifiers.items()
])
display(test_results_all)

=== TEST SET (final, held-out) ===
  classifier: nearest_centroid
  accuracy: 0.7724719101123596
  precision: 0.7106227106227107
  recall: 0.9897959183673469
  f1: 0.8272921108742004
  leaked_through (FP): 79
  wrongly_archived (FN): 2
  n: 356

For reference, all three on the test set:


,classifier,accuracy,precision,recall,f1,leaked_through (FP),wrongly_archived (FN),n
0,nearest_centroid,0.772472,0.710623,0.989796,0.827292,79,2,356
1,knn,0.842697,0.821101,0.913265,0.864734,39,17,356
2,logistic_regression,0.870787,0.894737,0.867347,0.880829,20,26,356


## 6. Timing comparison

The LLM gate is measured (in the repo's benchmark) at **~9s/title on CPU**. Embedding + classification here should be dramatically faster — but note the honest caveat: this notebook times *embedding* on whatever device Colab gave you (likely GPU), while the production deployment is CPU-only. The GPU number tells you the ceiling; a CPU timing pass below tells you what to actually expect if this ships as-is.

In [25]:
n_test = len(test_df)
gpu_embed_per_title = embed_timings["test"] / n_test if n_test else 0

t0 = time.perf_counter()
_ = best_clf.predict(X_test)
classify_elapsed = time.perf_counter() - t0
classify_per_title = classify_elapsed / n_test if n_test else 0

print(f"Embedding ({device}):     {gpu_embed_per_title*1000:.2f} ms/title")
print(f"Classification ({BEST_CLASSIFIER_NAME}): {classify_per_title*1000:.3f} ms/title")
print(f"Total ({device}):         {(gpu_embed_per_title+classify_per_title)*1000:.2f} ms/title")
print(f"LLM gate (measured, CPU): 9000 ms/title")
print(f"Speedup vs LLM gate:      ~{9.0/(gpu_embed_per_title+classify_per_title):.0f}x  (on {device}, not yet CPU-verified)")

Embedding (cuda):     12.27 ms/title
Classification (nearest_centroid): 0.004 ms/title
Total (cuda):         12.28 ms/title
LLM gate (measured, CPU): 9000 ms/title
Speedup vs LLM gate:      ~733x  (on cuda, not yet CPU-verified)


In [26]:
# CPU sanity pass — what actually matters for the production deployment.
cpu_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device="cpu")
sample_titles = test_df["title"].tolist()[:50]  # a slice is enough to get a stable per-title number

t0 = time.perf_counter()
_ = cpu_model.encode(sample_titles)
cpu_elapsed = time.perf_counter() - t0
cpu_per_title = cpu_elapsed / len(sample_titles)

print(f"CPU embedding: {cpu_per_title*1000:.1f} ms/title (measured on a {len(sample_titles)}-title sample)")
print(f"Classification is effectively free on top of that ({classify_per_title*1000:.3f} ms/title).")
print(f"Realistic CPU speedup vs the 9s/title LLM gate: ~{9.0/cpu_per_title:.0f}x")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

CPU embedding: 1861.5 ms/title (measured on a 50-title sample)
Classification is effectively free on top of that (0.004 ms/title).
Realistic CPU speedup vs the 9s/title LLM gate: ~5x


## 7. Save the winning classifier

So it can be lifted into the backend later if this approach is approved — no need to re-run this notebook to get the reference vectors / model back.

In [27]:
import pickle

with open(f"title_gate_classifier_{BEST_CLASSIFIER_NAME}.pkl", "wb") as f:
    pickle.dump(best_clf, f)

print(f"Saved to title_gate_classifier_{BEST_CLASSIFIER_NAME}.pkl")
print("Download it from the Colab file browser (left sidebar) before the runtime resets.")

Saved to title_gate_classifier_nearest_centroid.pkl
Download it from the Colab file browser (left sidebar) before the runtime resets.


## 8. Summary — fill this in after running

- Best classifier: ___
- Test F1 / precision / recall vs. LLM gate: ___
- `wrongly_archived` count on test (the costly failure mode): ___
- Realistic CPU speedup vs. the LLM gate: ___
- Decision: replace / keep LLM gate / needs a bigger dataset first

Next step either way, per `docs/brainstorm-general-architecture.md`: if this looks promising, it still runs in **shadow mode** against the LLM gate (and, once it exists, the hand-labeled gold set) for a validation window before actually retiring anything in production.